# BigAlpha AI 因子提交版

本因子来自前期自动化候选搜索、滚动统计检验、风险中性诊断和候选筛选。

正式评估时，平台调用：

```python
main(datasources, start_date, end_date)
```

输出严格为 `date`、`instrument`、`factor` 三列。


## BigAlpha V1.9 Low-Vol Heavy


In [ ]:
def main(datasources, start_date, end_date):
    """
    BigAlpha AI 因子：lowvol_heavy

    强化低波动与短期反转，目标偏向提高 IC 和风险剔除后的 ICIR。
    """
    import numpy as np
    import pandas as pd
    import dai

    start_ts = pd.Timestamp(start_date).normalize()
    end_ts = pd.Timestamp(end_date).normalize()

    query_start = start_ts - pd.Timedelta(days=0)
    query_end = (
        end_ts
        + pd.Timedelta(days=1)
        - pd.Timedelta(seconds=1)
    )

    filters = {
        "date": [
            query_start.strftime("%Y-%m-%d %H:%M:%S"),
            query_end.strftime("%Y-%m-%d %H:%M:%S"),
        ]
    }

    factorlib = dai.query(
        """
        SELECT
            date,
            instrument,
            reversal_5,
            volatility_5,
            turn,
            net_active_buy_amount_main
        FROM bigalpha_2026_factorlib
        """,
        filters=filters,
        compression=True,
    ).df()

    pool = dai.query(
        """
        SELECT
            date,
            instrument
        FROM bigalpha_2026_instruments
        """,
        filters=filters,
        compression=True,
    ).df()

    factorlib = factorlib.copy()
    pool = pool.copy()

    factorlib["date"] = pd.to_datetime(
        factorlib["date"],
        errors="coerce",
    ).dt.normalize()

    pool["date"] = pd.to_datetime(
        pool["date"],
        errors="coerce",
    ).dt.normalize()

    factorlib["instrument"] = (
        factorlib["instrument"].astype(str)
    )
    pool["instrument"] = pool["instrument"].astype(str)

    factorlib = factorlib.dropna(
        subset=["date", "instrument"]
    ).drop_duplicates(
        ["date", "instrument"],
        keep="last",
    )

    pool = pool.dropna(
        subset=["date", "instrument"]
    ).drop_duplicates(
        ["date", "instrument"],
        keep="last",
    )

    data = pool.merge(
        factorlib,
        on=["date", "instrument"],
        how="left",
        validate="one_to_one",
    )

    data = data.sort_values(
        ["date", "instrument"],
        kind="mergesort",
    ).reset_index(drop=True)

    def cs_rank(column, direction=1.0):
        values = pd.to_numeric(
            data[column],
            errors="coerce",
        )

        rank_pct = values.groupby(
            data["date"],
            sort=False,
        ).rank(
            method="average",
            pct=True,
        )

        ranked = direction * (2.0 * rank_pct - 1.0)

        return ranked.replace(
            [np.inf, -np.inf],
            np.nan,
        ).fillna(0.0)

    def rerank(values):
        values = pd.Series(
            values,
            index=data.index,
            dtype="float64",
        )

        rank_pct = values.groupby(
            data["date"],
            sort=False,
        ).rank(
            method="average",
            pct=True,
        )

        return (
            2.0 * rank_pct - 1.0
        ).replace(
            [np.inf, -np.inf],
            np.nan,
        ).fillna(0.0)

    r_reversal = cs_rank("reversal_5", +1.0)
    r_lowvol = cs_rank("volatility_5", -1.0)
    r_lowturn = cs_rank("turn", -1.0)
    r_flow = cs_rank(
        "net_active_buy_amount_main",
        +1.0,
    )

    raw_signal = (
        0.40 * r_reversal
        + 0.40 * r_lowvol
        + 0.15 * r_lowturn
        + 0.05 * r_flow
    )

    data["factor"] = rerank(raw_signal)

    output = data[
        (data["date"] >= start_ts)
        & (data["date"] <= end_ts)
    ][["date", "instrument", "factor"]].copy()

    output["instrument"] = output["instrument"].astype(str)
    output["factor"] = pd.to_numeric(
        output["factor"],
        errors="coerce",
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    ).fillna(0.0).astype("float64")

    output = output.drop_duplicates(
        ["date", "instrument"],
        keep="last",
    ).sort_values(
        ["date", "instrument"],
        kind="mergesort",
    ).reset_index(drop=True)

    return output